# FIT4112 – Lab 3 | Notebook 01: Customer Feature Engineering

**Bài toán:** Bảng giao dịch có nhiều dòng cho cùng một khách. K-Means cần **một dòng đặc trưng cho mỗi khách hàng**.

Ta sẽ tạo **RFM**:
- **Recency:** số ngày kể từ lần mua gần nhất đến mốc tham chiếu (nhỏ = mua gần đây).
- **Frequency:** số hóa đơn mua hàng khác nhau (**không** phải số dòng).
- **Monetary:** tổng chi tiêu hợp lệ trong dữ liệu.

**Foundation (HTTT):** chạy lần lượt các cell và trả lời câu hỏi. **Applied (PTPM):** ngoài việc chạy, giải thích vì sao định nghĩa Frequency/Monetary phù hợp.

## Bước 1. Tải dữ liệu

Bộ demo là dữ liệu **giả lập** nhằm minh họa. Có một vài giao dịch không hợp lệ để người học kiểm tra tiền xử lý.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def find_project_root():
    current = Path.cwd().resolve()
    for directory in [current, *current.parents]:
        if (directory / "data" / "raw" / "retail_transactions_demo.csv").exists():
            return directory
    raise FileNotFoundError("Không tìm thấy repo FIT4112 Lab 3. Mở notebook từ repo hoặc notebooks/.")

ROOT = find_project_root()
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Thư mục dự án:", ROOT)

DATA_FILE = "retail_transactions_demo.csv"
REFERENCE_DATE = pd.Timestamp("2026-07-01")  # mốc cố định, không dùng ngày hiện tại
COUNTRY = "United Kingdom"
raw = pd.read_csv(
    RAW_DIR / DATA_FILE,
    dtype={"InvoiceNo": "string", "StockCode": "string", "CustomerID": "string"},
)
print("Số dòng gốc:", len(raw))
print("Số khách hàng có ID:", raw["CustomerID"].nunique())
display(raw.head(8))

## Bước 2. Làm sạch

- Hóa đơn bắt đầu bằng `C` là hóa đơn hủy.
- Giữ `Quantity > 0`, `UnitPrice > 0`.
- Bỏ khách hàng không có `CustomerID`.
- Chỉ lấy giao dịch tại UK; ngày mua phải hợp lệ và **trước** ngày tham chiếu.

**Câu hỏi:** Nếu tính Frequency bằng số dòng giao dịch, điều gì xảy ra với một hóa đơn có 5 loại sản phẩm?

In [ ]:
clean = raw.copy()
clean["Quantity"] = pd.to_numeric(clean["Quantity"], errors="coerce")
clean["UnitPrice"] = pd.to_numeric(clean["UnitPrice"], errors="coerce")
clean["InvoiceDate"] = pd.to_datetime(clean["InvoiceDate"], errors="coerce")
clean["CustomerID"] = clean["CustomerID"].str.strip()
valid = (
    clean["InvoiceNo"].notna()
    & ~clean["InvoiceNo"].str.startswith("C", na=False)
    & clean["CustomerID"].notna()
    & clean["CustomerID"].ne("")
    & clean["Quantity"].gt(0)
    & clean["UnitPrice"].gt(0)
    & clean["Country"].eq(COUNTRY)
    & clean["InvoiceDate"].notna()
    & clean["InvoiceDate"].lt(REFERENCE_DATE)
)
clean = clean.loc[valid].copy()
clean["Revenue"] = clean["Quantity"] * clean["UnitPrice"]
print(f"Trước làm sạch: {len(raw):,} dòng; sau làm sạch: {len(clean):,} dòng")
print("Số khách hàng:", clean["CustomerID"].nunique())
display(clean.head(5))

## Bước 3. Tính RFM theo khách hàng

**Recency** được tính theo ngày (phần thời gian trong ngày được bỏ qua). **Frequency** là số mã hóa đơn duy nhất của mỗi khách. **Monetary** là tổng Revenue của tất cả dòng giao dịch hợp lệ.

In [ ]:
rfm = clean.groupby("CustomerID").agg(
    LastPurchase=("InvoiceDate", "max"),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum"),
).reset_index()
rfm["Recency"] = (REFERENCE_DATE.normalize() - rfm["LastPurchase"].dt.normalize()).dt.days
rfm = rfm[["CustomerID", "Recency", "Frequency", "Monetary"]].sort_values("CustomerID").reset_index(drop=True)
assert rfm["CustomerID"].is_unique
assert rfm[["Recency", "Frequency", "Monetary"]].gt(0).all().all()
display(rfm.head())
display(rfm[["Recency", "Frequency", "Monetary"]].describe().round(2))

## Bước 4. Quan sát mức độ khác nhau giữa các đặc trưng

Ba biến có đơn vị và thang đo khác nhau. Đây là lý do ta sẽ chuẩn hóa trong Notebook 02.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, col in zip(axes, ["Recency", "Frequency", "Monetary"]):
    rfm[col].hist(ax=ax, bins=12)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## Bước 5. Lưu bảng khách hàng

Notebook 02 sẽ đọc `customer_rfm.csv`. Nếu máy không chạy được Notebook 01 ngay, có thể dùng bảng starter trong `data/starter/` trước.

In [ ]:
output = OUT_DIR / "customer_rfm.csv"
rfm.to_csv(output, index=False)
print("Đã lưu:", output)
print("Số khách hàng:", len(rfm))

## TODO / Câu hỏi tự đánh giá

1. Mỗi hàng trong `rfm` biểu diễn điều gì?
2. Khách hàng có Recency = 5 khác gì với Recency = 120?
3. Tại sao Frequency dùng `nunique(InvoiceNo)` thay vì `count()`?
4. Monetary thể hiện chi tiêu nào? Tại sao loại giao dịch hủy và âm?

**Tiếp theo:** Notebook 02 – K-Means Clustering.